# Baseline YOLO12-Small — Aceh Road Damage Detection 2026 (Multi-seed)

Baseline pretrained empat kelas: `D00`, `D10`, `D20`, dan `D40`. Alur: audit dataset → train → validation → test → ZIP.

Multi-seed: seed 42, 123, 456 dengan hyperparameter identik. Hasil per seed disimpan sebagai ZIP terpisah, lalu diringkas sebagai mean ± std.

In [ ]:
# 1. Environment. Aktifkan GPU dan Internet pada Kaggle.
import json, platform, subprocess, sys, warnings, zipfile
from collections import Counter
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.163'], check=True)

# Ray pada environment Kaggle kadang meminta trial ID meskipun training ini
# bukan Ray Tune. Abaikan hanya warning spesifik yang tidak memengaruhi YOLO.
warnings.filterwarnings(
    'ignore',
    message=r'`get_trial_id` is meant to only be called.*',
    category=UserWarning,
    module=r'ray\.train\._internal\.session',
)

import torch
import ultralytics
# Versi dikunci sama dengan run seed 42 lama agar seed 123/456 setara.
assert ultralytics.__version__ == '8.4.163', f'Ultralytics {ultralytics.__version__} != 8.4.163; restart session lalu jalankan ulang.'
from ultralytics import YOLO

WORKDIR = Path('/kaggle/working')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def log_section(title):
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Device      : {DEVICE}')
print(f'GPU         : {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'WARNING: GPU tidak aktif')

# Protokol run seed 42 lama: PyTorch 2.10.0+cu128 dan Tesla T4. Perbedaan tidak menghentikan run,
# tetapi wajib dicatat di log sesi (README_SOP.md).
if torch.__version__ != '2.10.0+cu128' or not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    print('WARNING: environment berbeda dari run seed 42 lama — catat di log sesi README_SOP.md.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 4.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

ENVIRONMENT
Python      : 3.13.15
PyTorch     : 2.11.0+cu128
Ultralytics : 8.4.163
Device      : 0
GPU         : Tesla T4


In [ ]:
# 2. Konfigurasi dan audit dataset sebelum training.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ardd26/datasets_here')
DATA_YAML = WORKDIR / 'ardd26_4class.yaml'
CLASS_NAMES = ['D00', 'D10', 'D20', 'D40']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS = 0, 2
# Multi-seed: hyperparameter identik untuk setiap seed; hanya SEED yang berbeda.
SEEDS = [42, 123, 456]
# Batas sesi Kaggle 12 jam. Bila tiga seed tidak muat dalam satu sesi, jalankan sebagian,
# misalnya RUN_SEEDS = [42] lalu [123, 456] pada sesi berikutnya.
# Seed 42 memakai hasil run lama (lihat README_SOP.md); sesi ini menjalankan:
RUN_SEEDS = [123, 456]
# Opsional: folder hasil sesi sebelumnya (Kaggle input) untuk agregasi mean ± std.
# '/kaggle/input' dicari rekursif: hasil seed 42 lama dan output sesi sebelumnya otomatis ikut.
EXTRA_RESULT_DIRS = ['/kaggle/input']
EXPERIMENT_NAME = 'yolo12s_original_pretrained_ardd26'
RUNS_DIR = WORKDIR / 'runs'

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset tidak ditemukan: {DATA_ROOT}')

def file_stems(folder, extensions):
    if not folder.is_dir():
        raise FileNotFoundError(f'Folder tidak ditemukan: {folder}')
    return {p.stem.casefold() for p in folder.iterdir() if p.is_file() and p.suffix.lower() in extensions}

dataset_report = {}
for split in ('train', 'val', 'test'):
    image_dir, label_dir = (DATA_ROOT / split / name for name in ('images', 'labels'))
    image_stems = file_stems(image_dir, IMAGE_EXTENSIONS)
    label_stems = file_stems(label_dir, {'.txt'})
    if image_stems != label_stems:
        raise ValueError(f'Pairing {split} gagal: images={len(image_stems)}, labels={len(label_stems)}')
    counts, empty_labels = Counter(), 0
    for label_path in sorted(label_dir.glob('*.txt')):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        empty_labels += int(not lines)
        for line_number, line in enumerate(lines, 1):
            parts = line.split()
            if len(parts) != 5:
                raise ValueError(f'{label_path}:{line_number}: format bukan 5 kolom')
            class_id, coords = int(parts[0]), [float(v) for v in parts[1:]]
            if class_id not in range(4):
                raise ValueError(f'{label_path}:{line_number}: class ID {class_id} di luar 0..3')
            if not all(0 <= v <= 1 for v in coords) or coords[2] <= 0 or coords[3] <= 0:
                raise ValueError(f'{label_path}:{line_number}: bounding box invalid')
            counts[class_id] += 1
    dataset_report[split] = {'pairs': len(image_stems), 'objects': sum(counts.values()), 'empty_labels': empty_labels, 'classes': {CLASS_NAMES[i]: counts[i] for i in range(4)}}

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 4
names:
  0: D00
  1: D10
  2: D20
  3: D40
''', encoding='utf-8')

log_section('DATASET AUDIT PASSED')
print(json.dumps(dataset_report, indent=2))
print(f'Total pairs: {sum(v["pairs"] for v in dataset_report.values())}')
print(f'Total objects: {sum(v["objects"] for v in dataset_report.values())}')
print(DATA_YAML.read_text())

# Pemeriksaan awal (sebelum training berjam-jam): seed yang sudah punya hasil vs seed sesi ini.
PREVIOUS_SEEDS = sorted({
    json.loads(p.read_text(encoding='utf-8'))['seed']
    for d in EXTRA_RESULT_DIRS if Path(d).is_dir()
    for p in Path(d).rglob(f'{EXPERIMENT_NAME}_seed*_evaluation_metrics.json')
})
print(f'Seed dengan hasil di input : {PREVIOUS_SEEDS}')
print(f'Seed dijalankan sesi ini   : {RUN_SEEDS}')
if 42 not in PREVIOUS_SEEDS and 42 not in RUN_SEEDS:
    raise FileNotFoundError('Hasil seed 42 lama tidak ditemukan. Add Input dataset seed42-previous (README_SOP.md langkah A).')
if set(PREVIOUS_SEEDS) & set(RUN_SEEDS):
    raise ValueError(f'Seed {sorted(set(PREVIOUS_SEEDS) & set(RUN_SEEDS))} sudah punya hasil; '
                     'hapus dari RUN_SEEDS agar kuota GPU tidak terbuang.')
pending = sorted(set(SEEDS) - set(PREVIOUS_SEEDS) - set(RUN_SEEDS))
if pending:
    print(f'Catatan: seed {pending} dijadwalkan untuk sesi berikutnya.')


DATASET AUDIT PASSED
{
  "train": {
    "pairs": 4211,
    "objects": 7394,
    "empty_labels": 0,
    "classes": {
      "D00": 1974,
      "D10": 555,
      "D20": 1717,
      "D40": 3148
    }
  },
  "val": {
    "pairs": 526,
    "objects": 916,
    "empty_labels": 0,
    "classes": {
      "D00": 242,
      "D10": 69,
      "D20": 214,
      "D40": 391
    }
  },
  "test": {
    "pairs": 527,
    "objects": 928,
    "empty_labels": 0,
    "classes": {
      "D00": 252,
      "D10": 67,
      "D20": 212,
      "D40": 397
    }
  }
}
Total pairs: 5264
Total objects: 9238
path: /kaggle/input/datasets/danialalfayyadh/ardd26/datasets_here
train: train/images
val: val/images
test: test/images

nc: 4
names:
  0: D00
  1: D10
  2: D20
  3: D40

Seed dengan hasil di input : [42]
Seed dijalankan sesi ini   : [123, 456]


In [ ]:
# 3. Satu run per seed: train → validation → test → ZIP (isi sama dengan notebook seed 42).
import gc, shutil, statistics

def metric_summary(metrics):
    return {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'map50_95_per_class': {CLASS_NAMES[i]: float(v) for i, v in enumerate(metrics.box.maps)},
        'save_dir': str(metrics.save_dir),
    }

def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def seed_paths(seed):
    name = f'{EXPERIMENT_NAME}_seed{seed}'
    return (name, WORKDIR / f'{name}_evaluation_metrics.json',
            WORKDIR / f'{name}_config.json', WORKDIR / f'{name}_results.zip')

def write_seed_zip(zip_path, archive_items):
    tmp_path = zip_path.with_suffix('.zip.tmp')
    files = []
    for item in archive_items:
        files.extend([item] if item.is_file() else [p for p in item.rglob('*') if p.is_file()])
    with zipfile.ZipFile(tmp_path, 'w', zipfile.ZIP_DEFLATED) as archive:
        for file_path in files:
            archive.write(file_path, file_path.relative_to(WORKDIR))
    tmp_path.replace(zip_path)  # ZIP final = penanda seed selesai.
    return len(files)

def prepare_seed(seed):
    name, evaluation_json, _, zip_path = seed_paths(seed)
    if zip_path.is_file() and evaluation_json.is_file():
        log_section(f'SEED {seed} SUDAH SELESAI — DILEWATI')
        return json.loads(evaluation_json.read_text(encoding='utf-8'))
    # Hapus sisa run seed ini yang terputus agar exist_ok=False tetap valid.
    for suffix in ('', '_val', '_test'):
        shutil.rmtree(RUNS_DIR / f'{name}{suffix}', ignore_errors=True)
    return None

def run_seed(SEED):
    name, evaluation_json, run_config, zip_path = seed_paths(SEED)
    done = prepare_seed(SEED)
    if done is not None:
        return done

    log_section(f'TRAINING — YOLO12S PRETRAINED — SEED {SEED}')
    model = YOLO('yolo12s.pt')
    model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS,
        device=DEVICE, workers=WORKERS, project=str(RUNS_DIR), name=name,
        exist_ok=False, pretrained=True, optimizer=OPTIMIZER, lr0=LR0,
        momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, cos_lr=False,
        patience=PATIENCE, seed=SEED, deterministic=True, plots=True, verbose=True,
    )
    RUN_DIR = Path(model.trainer.save_dir)
    BEST_PT, LAST_PT = Path(model.trainer.best), Path(model.trainer.last)
    if not BEST_PT.is_file():
        raise FileNotFoundError(f'best.pt tidak ditemukan: {BEST_PT}')
    print(f'Run directory: {RUN_DIR}')
    print(f'Best weights : {BEST_PT}')
    print(f'Last weights : {LAST_PT}')
    del model
    free_gpu_memory()

    best_model = YOLO(str(BEST_PT))
    reports = {}
    for split in ('val', 'test'):
        log_section(f'{split.upper()} — BEST.PT — SEED {SEED}')
        metrics = best_model.val(
            data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH,
            device=DEVICE, workers=WORKERS, project=str(RUNS_DIR),
            name=f'{name}_{split}', exist_ok=False, plots=True,
        )
        reports[split] = metric_summary(metrics)
        print(json.dumps(reports[split], indent=2))
        del metrics
    del best_model
    free_gpu_memory()

    EVALUATION_REPORT = {'seed': SEED, 'validation': reports['val'], 'test': reports['test']}
    evaluation_json.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
    run_config.write_text(json.dumps({
        'model': 'yolo12s.pt', 'dataset_root': str(DATA_ROOT), 'classes': CLASS_NAMES,
        'dataset_report': dataset_report, 'epochs': EPOCHS, 'imgsz': IMGSZ,
        'batch': BATCH, 'nbs': NBS, 'optimizer': OPTIMIZER, 'lr0': LR0,
        'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED, 'seeds': SEEDS,
        'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
    }, indent=2), encoding='utf-8')
    n_files = write_seed_zip(zip_path, [
        RUN_DIR, Path(reports['val']['save_dir']), Path(reports['test']['save_dir']),
        DATA_YAML, run_config, evaluation_json,
    ])
    log_section(f'SEED {SEED} COMPLETED')
    print(json.dumps(EVALUATION_REPORT, indent=2))
    print(f'ZIP created: {zip_path} ({n_files} files)')
    return EVALUATION_REPORT

In [ ]:
# 4. Jalankan semua seed secara berurutan; memori GPU dibersihkan setelah setiap seed.
ALL_REPORTS = {}
for seed in RUN_SEEDS:
    ALL_REPORTS[seed] = run_seed(seed)
    free_gpu_memory()
    if torch.cuda.is_available():
        print(f'GPU memory setelah seed {seed}: {torch.cuda.memory_allocated() / 1024**2:.0f} MiB allocated')


TRAINING — YOLO12S PRETRAINED — SEED 123
New https://pypi.org/project/ultralytics/8.4.176 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/ardd26_4class.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=160, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, 

In [ ]:
# 5. Agregasi mean ± std seluruh seed yang sudah selesai, lalu ZIP ringkasan.
pattern = f'{EXPERIMENT_NAME}_seed*_evaluation_metrics.json'
found = {}
for path in sorted(WORKDIR.glob(pattern)) + [p for d in EXTRA_RESULT_DIRS for p in sorted(Path(d).rglob(pattern))]:
    report = json.loads(path.read_text(encoding='utf-8'))
    found.setdefault(report['seed'], (path, report))

def mean_std(values):
    return {'mean': statistics.mean(values), 'std': statistics.stdev(values) if len(values) > 1 else 0.0,
            'values': values}

METRIC_KEYS = ('precision', 'recall', 'map50', 'map50_95')
SUMMARY = {'experiment': EXPERIMENT_NAME, 'seeds': sorted(found)}
for split_key in ('validation', 'test'):
    reports = [found[s][1][split_key] for s in sorted(found)]
    if not reports:
        continue
    SUMMARY[split_key] = {k: mean_std([r[k] for r in reports]) for k in METRIC_KEYS}
    SUMMARY[split_key]['map50_95_per_class'] = {
        c: mean_std([r['map50_95_per_class'][c] for r in reports]) for c in CLASS_NAMES
    }

SUMMARY_JSON = WORKDIR / f'{EXPERIMENT_NAME}_multiseed_summary.json'
SUMMARY_JSON.write_text(json.dumps(SUMMARY, indent=2), encoding='utf-8')
SUMMARY_CSV = WORKDIR / f'{EXPERIMENT_NAME}_multiseed_summary.csv'
rows = ['seed,split,' + ','.join(METRIC_KEYS)]
for s in sorted(found):
    for split_key in ('validation', 'test'):
        r = found[s][1][split_key]
        rows.append(f'{s},{split_key},' + ','.join(f'{r[k]:.6f}' for k in METRIC_KEYS))
for split_key in ('validation', 'test'):
    if split_key in SUMMARY:
        for stat in ('mean', 'std'):
            rows.append(f'{stat},{split_key},' + ','.join(f'{SUMMARY[split_key][k][stat]:.6f}' for k in METRIC_KEYS))
SUMMARY_CSV.write_text('\n'.join(rows) + '\n', encoding='utf-8')

SUMMARY_ZIP = WORKDIR / f'{EXPERIMENT_NAME}_multiseed_summary.zip'
with zipfile.ZipFile(SUMMARY_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in [SUMMARY_JSON, SUMMARY_CSV, *(found[s][0] for s in sorted(found))]:
        archive.write(path, path.name)

log_section(f'MULTI-SEED SUMMARY — {len(found)}/{len(SEEDS)} SEED')
missing = sorted(set(SEEDS) - set(found))
if missing:
    print(f'WARNING: seed belum selesai: {missing}')
print(SUMMARY_CSV.read_text(encoding='utf-8'))
from IPython.display import FileLink, display
display(FileLink(SUMMARY_ZIP))
for s in sorted(found):
    seed_zip = seed_paths(s)[3]
    if seed_zip.is_file():
        display(FileLink(seed_zip))


MULTI-SEED SUMMARY — 3/3 SEED
seed,split,precision,recall,map50,map50_95
42,validation,0.704769,0.650274,0.729358,0.462555
42,test,0.690845,0.706835,0.753887,0.497375
123,validation,0.712017,0.704368,0.749302,0.476830
123,test,0.739137,0.654520,0.761261,0.488699
456,validation,0.685520,0.669824,0.734047,0.474745
456,test,0.738184,0.690491,0.754761,0.499440
mean,validation,0.700768,0.674822,0.737569,0.471377
std,validation,0.013694,0.027391,0.010428,0.007711
mean,test,0.722722,0.683949,0.756636,0.495171
std,test,0.027611,0.026764,0.004029,0.005700



/kaggle/working/yolo12s_original_pretrained_ardd26_multiseed_summary.zip

/kaggle/working/yolo12s_original_pretrained_ardd26_seed123_results.zip

/kaggle/working/yolo12s_original_pretrained_ardd26_seed456_results.zip